# TTA Example

## Imports and Configs

In [1]:
import sys
from os import path, environ
from argparse import ArgumentParser

import torch
from torchinfo import summary

from ttadapters import datasets, models, methods
from ttadapters.utils import visualizer, validator
from ttadapters.datasets import scenarios

from ttadapters.methods.regularizers.temporal.apt.gate_engine import APTConfidenceGatedConfig, APTConfidenceGatedEngine

In [2]:
import pandas as pd

pd.options.display.float_format = lambda x: f"{x*100 if x < 1 else x:.4f}"

In [3]:
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

### Parse Arguments

In [4]:
# Set Batch Size
BATCH_SIZE = 1  # Online

# Set Total Rounds
TOTAL_ROUNDS = 10

# Set Data Root
DATA_ROOT = path.join(".", "data")

# Set Target Dataset
SOURCE_DOMAIN = datasets.SHIFTDataset

# Set Model List
MODEL_ZOO = ["rcnn", "swinrcnn", "yolo11", "rtdetr"]
MODEL_TYPE = MODEL_ZOO[0]

In [5]:
# Create argument parser
parser = ArgumentParser(description="Adaptation experiment script for Test-Time Adapters")

# Add model arguments
parser.add_argument("--dataset", type=str, choices=["shift", "city"], default="shift", help="Training dataset")
parser.add_argument("--model", type=str, choices=MODEL_ZOO, default=MODEL_TYPE, help="Model architecture")

# Add training arguments
parser.add_argument("--adapt-batch", type=int, default=BATCH_SIZE, help="Adaptation batch size")
parser.add_argument("--total-rounds", type=int, default=TOTAL_ROUNDS, help="Total rounds")
parser.add_argument("--data-root", type=str, default=DATA_ROOT, help="Root directory for datasets")
parser.add_argument("--device", type=int, default=0, help="CUDA device number")
parser.add_argument("--additional_gpu", type=int, default=0, help="Additional CUDA device count")
parser.add_argument("--use-bf16", action="store_true", help="Use bfloat16 precision")

# Parsing arguments
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
    print("INFO: Running in notebook mode with default arguments")
else:
    args = parser.parse_args()

# Update global variables based on parsed arguments
BATCH_SIZE = args.adapt_batch
TOTAL_ROUNDS = args.total_rounds
DATA_ROOT = args.data_root
MODEL_TYPE = args.model
match args.dataset:
    case "shift":
        SOURCE_DOMAIN = datasets.SHIFTDataset
    case "city":
        SOURCE_DOMAIN = datasets.CityScapesDataset
    case _:
        raise ValueError(f"Unsupported dataset: {args.dataset}")
print(f"INFO: Running online adaptation with batch size {BATCH_SIZE}")

INFO: Running in notebook mode with default arguments
INFO: Running online adaptation with batch size 1


### Check GPU Availability

In [6]:
!nvidia-smi

Tue Dec 23 03:51:19 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.161.07             Driver Version: 535.161.07   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          On  | 00000000:65:00.0 Off |                   On |
| N/A   60C    P0             153W / 300W |                  N/A |     N/A      Default |
|                                         |                      |              Enabled |
+-----------------------------------------+----------------------+--

In [7]:
# Set CUDA Device Number
DEVICE_NUM = 0 if not args.device else args.device
ADDITIONAL_GPU = 0 if not args.additional_gpu else args.additional_gpu
DATA_TYPE = torch.float32 if not args.use_bf16 else torch.bfloat16

if torch.cuda.is_available():
    if ADDITIONAL_GPU:
        torch.cuda.set_device(DEVICE_NUM)
        device = torch.device("cuda")
    else:
        device = torch.device(f"cuda:{DEVICE_NUM}")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1

print(f"INFO: Using device - {device}" + (f":{DEVICE_NUM}" if ADDITIONAL_GPU else ""))
print(f"INFO: Using data precision - {DATA_TYPE}")

INFO: Using device - cuda:0
INFO: Using data precision - torch.float32


## Define Dataset

In [8]:
# Fast download patch
datasets.patch_fast_download_for_object_detection()

In [9]:
# Dataset info
CLASSES = datasets.SHIFTClearDatasetForObjectDetection.classes
NUM_CLASSES = len(CLASSES)
print(f"INFO: Number of classes - {NUM_CLASSES} {CLASSES}")

INFO: Number of classes - 6 ['pedestrian', 'car', 'truck', 'bus', 'motorcycle', 'bicycle']


## Load Base Model

In [10]:
# Initialize base_model
match MODEL_TYPE:
    case "rcnn":
        base_model = models.FasterRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "swinrcnn":
        base_model = models.SwinRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "yolo11":
        # DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.YOLO11ForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "rtdetr":
        DATA_TYPE = torch.bfloat16  # bf16 default
        base_model = models.RTDetrForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case _:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")

print("INFO: Model state loaded -", load_result)
base_model.to(device)

INFO: Model state loaded - <All keys matched successfully>


FasterRCNNForObjectDetection(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2): Sequential(
        (0): Bo

In [11]:
summary(base_model)

Layer (type:depth-idx)                                  Param #
FasterRCNNForObjectDetection                            --
├─FPN: 1-1                                              --
│    └─Conv2d: 2-1                                      65,792
│    └─Conv2d: 2-2                                      590,080
│    └─Conv2d: 2-3                                      131,328
│    └─Conv2d: 2-4                                      590,080
│    └─Conv2d: 2-5                                      262,400
│    └─Conv2d: 2-6                                      590,080
│    └─Conv2d: 2-7                                      524,544
│    └─Conv2d: 2-8                                      590,080
│    └─LastLevelMaxPool: 2-9                            --
│    └─ResNet: 2-10                                     --
│    │    └─BasicStem: 3-1                              (9,408)
│    │    └─Sequential: 3-2                             (212,992)
│    │    └─Sequential: 3-3                             1,2

### Load Scenarios

In [12]:
# Ensure split (required due to Scenario class works with coroutines)
_ = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)

[12/23/2025 03:51:23] SHIFT DevKit - INFO - Base: ./data/SHIFT/continuous/images/1x/train. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:23] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/daytime_to_night/continuous/images/1x/train/front/det_2d.json' ...


INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/continuous/1x...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/continuous/1x...
INFO: Dataset archive found in the root directory. Skipping download.


[12/23/2025 03:51:23] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/daytime_to_night/continuous/images/1x/train/front/det_2d.json' Done.
[12/23/2025 03:51:24] SHIFT DevKit - INFO - Loading annotation takes 1.09 seconds.


Batch 0:

Item                 Shape                               Min        Max       
--------------------------------------------------------------------------------
original_hw          [tensor([800]), tensor([1280])]
input_hw             [tensor([800]), tensor([1280])]
frame_ids            torch.Size([1])                           0.00       0.00
name                 ['00000000_img_front.jpg']
videoName            ['0039-134e']
intrinsics           torch.Size([1, 3, 3])                     0.00     640.00
extrinsics           torch.Size([1, 4, 4])                  -191.55      57.56
boxes2d              torch.Size([1, 3, 4])                   112.00     494.00
boxes2d_classes      torch.Size([1, 3])                        0.00       0.00
boxes2d_track_ids    torch.Size([1, 3])                        0.00       2.00
images               torch.Size([1, 3, 800, 1280])             0.00     255.00

Video name: 0039-134e
Sample indices within a video: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10,

In [13]:
data_preparation = base_model.DataPreparation(datasets.base.BaseDataset(), evaluation_mode=True)

match SOURCE_DOMAIN:
    case datasets.SHIFTDataset:
        discrete_scenario = scenarios.SHIFTDiscreteScenario(
            root=DATA_ROOT, valid=True, order=scenarios.SHIFTDiscreteScenario.WHWPAPER, transforms=data_preparation.transforms
        )
    case datasets.CityScapesDataset:
        discrete_scenario = None
        continuous_scenario = None
    case _:
        raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.


[12/23/2025 03:51:24] SHIFT DevKit - INFO - Base: ./data/SHIFT/discrete/images/val. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:24] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/cloudy_daytime/discrete/images/val/front/det_2d.json' ...
[12/23/2025 03:51:25] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/cloudy_daytime/discrete/images/val/front/det_2d.json' Done.
[12/23/2025 03:51:25] SHIFT DevKit - INFO - Loading annotation takes 0.85 seconds.
[12/23/2025 03:51:25] SHIFT DevKit - INFO - Base: ./data/SHIFT/discrete/images/val. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:25] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/overcast_daytime/discrete/images/val/front/det_2d.json' ...


INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.


[12/23/2025 03:51:26] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/overcast_daytime/discrete/images/val/front/det_2d.json' Done.
[12/23/2025 03:51:26] SHIFT DevKit - INFO - Loading annotation takes 0.66 seconds.
[12/23/2025 03:51:26] SHIFT DevKit - INFO - Base: ./data/SHIFT/discrete/images/val. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:26] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/foggy_daytime/discrete/images/val/front/det_2d.json' ...


INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.


...

[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation takes 0.66 seconds.
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Base: ./data/SHIFT/discrete/images/val. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/clear_night/discrete/images/val/front/det_2d.json' ...
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/clear_night/discrete/images/val/front/det_2d.json' Done.


INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.


[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation takes 0.37 seconds.
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Base: ./data/SHIFT/discrete/images/val. Backend: <shift_dev.utils.backend.ZipBackend object at 0x7f6094b66650>
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/clear_daytime/discrete/images/val/front/det_2d.json' ...
[12/23/2025 03:51:29] SHIFT DevKit - INFO - Loading annotation from './data/SHIFT_SUBSET/clear_daytime/discrete/images/val/front/det_2d.json' Done.


INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.
INFO: Subset split for 'SHIFT_SUBSET' dataset is already done. Skipping...
INFO: Downloading 'SHIFT_SUBSET' from file server to ./data/SHIFT/discrete...
INFO: Dataset archive found in the root directory. Skipping download.


[12/23/2025 03:51:30] SHIFT DevKit - INFO - Loading annotation takes 0.95 seconds.


## Test Frequency-Based Normalizer

In [15]:
# Import frequency normalizer
from frequency.not_heuristic_apt_frequencyv3 import FrequencyAdaptationEngine, FrequencyAdaptationConfig
import torch.nn as nn

config = FrequencyAdaptationConfig(
    adaptation_name="FrequencyTTAEngine_v3",
    hidden_dim=16,
    temperature=0.01,
    adapt_lr=1e-4,
    optim="SGD"
)

freq_tta = FrequencyAdaptationEngine(base_model, config)
freq_tta.to(device)
freq_tta.online()




FrequencyTTAEngine(
  (base_model): FasterRCNNForObjectDetection(
    (backbone): FPN(
      (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
      (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
      (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
      (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
      (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (top_block): LastLevelMaxPool()
      (bottom_up): ResNet(
        (stem): BasicStem(
          (conv1): Conv2d(
            3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
            (norm): FrozenBatchNorm2d(num_features=64, eps=

### Setup TTA with Frequency Normalization

In [16]:
# Create MethodContainer with both models
tta= methods.MethodContainer(**{
    # 'base model': base_model,
    'FreqTTA': freq_tta
})
print(f"Methods: {tta.names()}")

Methods: ('FreqTTA Round 1',)


### Evaluate with Frequency Normalization

In [17]:
evaluator = validator.DetectionEvaluator(tta.methods(), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False)
evaluator_loader_params = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

adaptation_results = []
for this in range(TOTAL_ROUNDS):
    result = visualizer.visualize_metrics(discrete_scenario(**evaluator_loader_params).play(evaluator, index=tta.names()))
    adaptation_results.append(result)


Output()

SHIFT Discrete Scenario:   0%|          | 0/7 [00:00<?, ?it/s]

Evaluation for cloudy_daytime:   0%|          | 0/2400 [00:00<?, ?it/s]

Evaluation for overcast_daytime:   0%|          | 0/1600 [00:00<?, ?it/s]

Evaluation for foggy_daytime:   0%|          | 0/2650 [00:00<?, ?it/s]

...

Output()

SHIFT Discrete Scenario:   0%|          | 0/7 [00:00<?, ?it/s]

Evaluation for cloudy_daytime:   0%|          | 0/2400 [00:00<?, ?it/s]


Evaluation interrupted by user


KeyboardInterrupt: 

In [ ]:
# # 1. 모델 로드 (fix_bn_stats=False로 COCO 대체 방지)
# base_model = models.YOLO11ForObjectDetection(dataset=SOURCE_DOMAIN)
# load_result =base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR),
# strict=False, fix_bn_stats=False)
# base_model.to(device)

# # 2. SHIFT Clear 데이터로 BN stats calibration
# from torch.utils.data import DataLoader

# clear_dataset =datasets.SHIFTClearDatasetForObjectDetection(root=DATA_ROOT, train=True)
# data_prep = base_model.DataPreparation(clear_dataset, evaluation_mode=True)
# clear_loader = DataLoader(clear_dataset, batch_size=16, shuffle=False, collate_fn=data_prep.collate_fn)

# # Calibration (한 번만 실행하면 캐시에 저장됨)
# base_model.calibrate_bn_stats(clear_loader, num_batches=100, device=device)

# # 3. 이후 평가 진행
# base_model.eval()